### Refactored A/B testing - clean up process

In [1]:
!rm -rf InkubaLM-Challenge
!git clone https://github.com/melissafasol/InkubaLM-Challenge.git
%cd InkubaLM-Challenge


Cloning into 'InkubaLM-Challenge'...
remote: Enumerating objects: 207, done.
remote: Counting objects: 100% (5/5), done.
remote: Compressing objects: 100% (4/4), done.
remote: Total 207 (delta 0), reused 3 (delta 0), pack-reused 202 (from 1)
Receiving objects: 100% (207/207), 907.92 KiB | 15.65 MiB/s, done.
Resolving deltas: 100% (127/127), done.
/content/InkubaLM-Challenge


In [2]:

!git checkout refactor-src-structure
!git pull

Branch 'refactor-src-structure' set up to track remote branch 'refactor-src-structure' from 'origin'.
Switched to a new branch 'refactor-src-structure'
Already up to date.


In [3]:
!pip install datasets
!pip install transformers datasets peft trl accelerate bitsandbytes



In [6]:
import sys
sys.path.append("..")  # Add parent directory to the path

import os
from typing import List
from pathlib import Path
import numpy as np

# DO NOT EDIT
# create submission file
import pandas as pd
from huggingface_hub import login
from transformers import (
    AutoTokenizer,
)
from src import (
    data_utils,
    model_utils,
    inference,
    prompts,
    evaluation,
    )

import torch
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, PeftModel
from datasets import load_dataset, concatenate_datasets, Dataset, Value, DatasetDict

from trl import SFTConfig, SFTTrainer, DataCollatorForCompletionOnlyLM
from peft import PeftModel, PeftConfig
from sklearn.model_selection import train_test_split




In [7]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [16]:
df

Dataset({
    features: ['ID', 'langs', 'instruction', 'inputs', 'targets'],
    num_rows: 1400
})

In [17]:
df = data_utils.load_and_combine_datasets(tag="Train", split="train")
df = df.map(lambda x: {"task": data_utils.extract_task_from_id(x["ID"])})


Common Columns: ['langs', 'ID', 'inputs', 'targets', 'instruction']


Map:   0%|          | 0/1400 [00:00<?, ? examples/s]

In [22]:
df

Dataset({
    features: ['ID', 'langs', 'instruction', 'inputs', 'targets', 'task'],
    num_rows: 1400
})

In [24]:
df_pd = df.to_pandas()

In [9]:

model_name = "lelapa/InkubaLM-0.4B"
model, tokenizer, bnb_config = model_utils.setup_model_and_tokenizer(model_name)

model = model_utils.apply_lora_adapters(model)


Common Columns: ['langs', 'ID', 'inputs', 'targets', 'instruction']


tokenizer_config.json:   0%|          | 0.00/960 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/991k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/3.95M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/763 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/2.66G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/111 [00:00<?, ?B/s]

trainable params: 524,288 || all params: 664,684,544 || trainable%: 0.0789


In [25]:
balanced_df = data_utils.balance_target_lengths(df_pd, repetition_factor=None, tokenizer=tokenizer.tokenize)

In [ ]:
from src.model_loader import setup_model_and_tokenizer, apply_lora_adapters

model_name = "lelapa/InkubaLM-0.4B"
model, tokenizer, _ = setup_model_and_tokenizer(model_name)
model = apply_lora_adapters(model)
